In [ ]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
# imports

In [ ]:
# ============================================================
# Imports
# ============================================================


# Import libraries
# Transformers imports classes from the Hugging Face Transformers linrary
# Autotokenizer converts human readable text into tokens that the model understands
#  AutoModelForSeq2SeqLM loads medical mt5 model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# import huggingface dataset library
from datasets import load_dataset
from datasets import Dataset
from datasets import concatenate_datasets

from transformers import Trainer
from transformers import TrainingArguments
from transformers import DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
# ============================================================
# Load datasets
# ============================================================


# load Enlgish dataset
# load dataste from huggingface, english
ds = load_dataset("HiTZ/MedExpQA", "en")
dataset = ds["train"]

ds_es = load_dataset("HiTZ/MedExpQA", "es")
spanish_dataset = ds_es["train"]

ds_fr = load_dataset("HiTZ/MedExpQA", "fr")
french_dataset = ds_fr["train"]

ds_it = load_dataset("HiTZ/MedExpQA", "it")
italian_dataset = ds_it["train"]

# ============================================================
# Prepare English dataset
# ============================================================


dataset = dataset.shuffle(seed=42)

first_split = dataset.train_test_split(train_size=0.70, seed = 42)

# Extract two datasets
train_dataset = first_split["train"]
remaining_dataset = first_split["test"] # This should be divided between validdation and test
remaining_split = remaining_dataset.train_test_split(train_size = 17/30, seed = 42)

#Extract datasets
validation_dataset = remaining_split["train"]
test_dataset = remaining_split["test"]


# ============================================================
# Create English ID sets
# ============================================================


# sets
training_ids = set()

for i in range(len(train_dataset)):
        example = train_dataset[i]
        question_id = example["id"]
        training_ids.add(question_id)

validation_ids = set()

for i in range(len(validation_dataset)):
        example = validation_dataset[i]
        question_id = example["id"]
        validation_ids.add(question_id)

test_ids = set()

for i in range(len(test_dataset)):
        example = test_dataset[i]
        question_id = example["id"]
        test_ids.add(question_id)

print("")
print(len(training_ids))
print(len(validation_ids))
print(len(test_ids))

# ============================================================
# Create Spanish splits
# ============================================================
spanish_train = []
spanish_validation = []
spanish_test = []

for i in range(len(spanish_dataset)):

    example = spanish_dataset[i]

    question_id = example["id"]

    if question_id in training_ids:
        spanish_train.append(example)

    elif question_id in validation_ids:
        spanish_validation.append(example)

    elif question_id in test_ids:
        spanish_test.append(example)

print("")
print("Spanish Training:", len(spanish_train))
print("Spanish Validation:", len(spanish_validation))
print("Spanish Test:", len(spanish_test))

# ============================================================
# Create French splits
# ============================================================

french_train = []
french_validation = []
french_test = []

for i in range(len(french_dataset)):

    example = french_dataset[i]

    question_id = example["id"]

    if question_id in training_ids:
        french_train.append(example)

    elif question_id in validation_ids:
        french_validation.append(example)

    elif question_id in test_ids:
        french_test.append(example)

print("")
print("French Training:", len(french_train))
print("French Validation:", len(french_validation))
print("French Test:", len(french_test))


# ============================================================
# Create Italian splits
# ============================================================

italian_train = []
italian_validation = []
italian_test = []

for i in range(len(italian_dataset)):

    example = italian_dataset[i]

    question_id = example["id"]

    if question_id in training_ids:
        italian_train.append(example)

    elif question_id in validation_ids:
        italian_validation.append(example)

    elif question_id in test_ids:
        italian_test.append(example)

print("")
print("Italian Training:", len(italian_train))
print("Italian Validation:", len(italian_validation))
print("Italian Test:", len(italian_test))



#--------------------------------------------------------------
# DATASETS
#-------------------------------------------------------------

spanish_train = Dataset.from_list(spanish_train)
spanish_validation = Dataset.from_list(spanish_validation)
spanish_test = Dataset.from_list(spanish_test)

french_train = Dataset.from_list(french_train)
french_validation = Dataset.from_list(french_validation)
french_test = Dataset.from_list(french_test)

italian_train = Dataset.from_list(italian_train)
italian_validation = Dataset.from_list(italian_validation)
italian_test = Dataset.from_list(italian_test)


303
74
57

Spanish Training: 303
Spanish Validation: 74
Spanish Test: 57

French Training: 303
French Validation: 74
French Test: 57

Italian Training: 303
Italian Validation: 74
Italian Test: 57


In [ ]:
!pip uninstall -y torchao

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece peft

In [ ]:
#----------------------------
# Load the tokenizer and model
#----------------------------

model_name = "HiTZ/Medical-mT5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Patch the compatibility issue between newer peft and transformers versions
if not hasattr(model, "_prepare_encoder_decoder_kwargs_for_generation"):
    model._prepare_encoder_decoder_kwargs_for_generation = lambda *args, **kwargs: {}

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, lora_config)

model.enable_input_require_grads()

model.gradient_checkpointing_enable()

model.print_trainable_parameters()

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 2,359,296 || all params: 1,231,940,608 || trainable%: 0.1915


In [ ]:
#----------------------------
# Preprocessing function
#----------------------------

def preprocess(example):

    question = example["full_question"]

    answer = example["full_answer"]


    prompt = f"""Question:
{question}

Answer:
"""



    model_inputs = tokenizer(
        prompt,
        max_length=512,
        truncation=True,

    )

    labels = tokenizer(
        text_target=answer,
        max_length=256,
        truncation=True,

    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
from datasets import load_dataset

dataset = load_dataset('HiTZ/MedExpQA', 'en')

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['correct_option', 'explanations', 'full_answer', 'full_answer_no_ref', 'full_question', 'id', 'lang', 'options', 'question_id_specific', 'type', 'year', 'rag'],
        num_rows: 434
    })
    validation: Dataset({
        features: ['correct_option', 'explanations', 'full_answer', 'full_answer_no_ref', 'full_question', 'id', 'lang', 'options', 'question_id_specific', 'type', 'year', 'rag'],
        num_rows: 63
    })
    test: Dataset({
        features: ['correct_option', 'explanations', 'full_answer', 'full_answer_no_ref', 'full_question', 'id', 'lang', 'options', 'question_id_specific', 'type', 'year', 'rag'],
        num_rows: 125
    })
})


In [ ]:
print(dataset["train"].features)

{'correct_option': Value('int64'), 'explanations': {'1': {'exist': Value('bool'), 'char_ranges': List(List(Value('int64'))), 'word_ranges': List(List(Value('int64'))), 'text': Value('string')}, '2': {'exist': Value('bool'), 'char_ranges': List(List(Value('int64'))), 'word_ranges': List(List(Value('int64'))), 'text': Value('string')}, '3': {'exist': Value('bool'), 'char_ranges': List(List(Value('int64'))), 'word_ranges': List(List(Value('int64'))), 'text': Value('string')}, '4': {'exist': Value('bool'), 'char_ranges': List(List(Value('int64'))), 'word_ranges': List(List(Value('int64'))), 'text': Value('string')}, '5': {'exist': Value('bool'), 'char_ranges': List(List(Value('int64'))), 'word_ranges': List(List(Value('int64'))), 'text': Value('string')}}, 'full_answer': Value('string'), 'full_answer_no_ref': Value('string'), 'full_question': Value('string'), 'id': Value('int64'), 'lang': Value('string'), 'options': {'1': Value('string'), '2': Value('string'), '3': Value('string'), '4': Va

In [ ]:
print(dataset["train"].column_names)

['correct_option', 'explanations', 'full_answer', 'full_answer_no_ref', 'full_question', 'id', 'lang', 'options', 'question_id_specific', 'type', 'year', 'rag']


In [ ]:
print(dataset["train"][0])

{'correct_option': 3, 'explanations': {'1': {'exist': False, 'char_ranges': [], 'word_ranges': [], 'text': ''}, '2': {'exist': False, 'char_ranges': [], 'word_ranges': [], 'text': ''}, '3': {'exist': True, 'char_ranges': [[429, 535]], 'word_ranges': [[66, 82]], 'text': 'The correct action implies inescapably the personal and direct contact between the doctor and the patient.'}, '4': {'exist': False, 'char_ranges': [], 'word_ranges': [], 'text': ''}, '5': {'exist': False, 'char_ranges': [], 'word_ranges': [], 'text': ''}}, 'full_answer': 'There is no doubt that the doctor-patient relationship today transcends the physical environment of the consultation room. But we must not forget that we must maintain the same ethical and professional codes as in the real environment. The code of ethics Article 26-3. The clinical practice of medicine by means of consultations exclusively by letter, telephone, radio, press or internet, is contrary to the deontological norms. The correct action implies 

In [ ]:
# Check that question contains no choices

In [ ]:
print(train_dataset[0]["full_question"])
print("\nREFERENCE ANSWER:")
print(train_dataset[0]["full_answer"])

In a 3-month-old infant all the clinical manifestations outlined below would make it necessary to rule out cystic fibrosis, EXCEPT:

REFERENCE ANSWER:
Dehydration may be due to diarrhea of other etiologies. I have doubts about answer 3 since it does not specify which secretions are involved. We have to understand that the growth of Pseudomona aeruginosa is in the culture of bronchial secretions. Because clearly the indication to perform a sweat chlorine (to rule out FQP) are the HIPOnatremic dehydrations and not the hypernatremic ones as the answer says.


In [ ]:
#----------------------------
# Test preprocessing
#----------------------------

sample = preprocess(train_dataset[0])

print("INPUT:")
print(tokenizer.decode(sample["input_ids"]))

print("\nTARGET:")
print(tokenizer.decode(sample["labels"]))

INPUT:
Question: In a 3-month-old infant all the clinical manifestations outlined below would make it necessary to rule out cystic fibrosis, EXCEPT: Answer:</s>

TARGET:
Dehydration may be due to diarrhea of other etiologies. I have doubts about answer 3 since it does not specify which secretions are involved. We have to understand that the growth of Pseudomona aeruginosa is in the culture of bronchial secretions. Because clearly the indication to perform a sweat chlorine (to rule out FQP) are the HIPOnatremic dehydrations and not the hypernatremic ones as the answer says.</s>


In [ ]:
#----------------------------
# Tokenize datasets
#----------------------------


english_train_tokenized = train_dataset.map(
    preprocess,
    remove_columns=train_dataset.column_names
)

english_validation_tokenized = validation_dataset.map(
    preprocess,
    remove_columns=validation_dataset.column_names
)

spanish_train_tokenized = spanish_train.map(
    preprocess,
    remove_columns=spanish_train.column_names
)

spanish_validation_tokenized = spanish_validation.map(
    preprocess,
    remove_columns=spanish_validation.column_names
)

french_train_tokenized = french_train.map(
    preprocess,
    remove_columns=french_train.column_names
)

french_validation_tokenized = french_validation.map(
    preprocess,
    remove_columns=french_validation.column_names
)

italian_train_tokenized = italian_train.map(
    preprocess,
    remove_columns=italian_train.column_names
)

italian_validation_tokenized = italian_validation.map(
    preprocess,
    remove_columns=italian_validation.column_names
)


train_tokenized = concatenate_datasets([
    english_train_tokenized,
    spanish_train_tokenized,
    french_train_tokenized,
    italian_train_tokenized
])

validation_tokenized = concatenate_datasets([
    english_validation_tokenized,
    spanish_validation_tokenized,
    french_validation_tokenized,
    italian_validation_tokenized
])

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

In [ ]:
#--------------------------
# How the model will learn
#--------------------------------

print("Total multilingual training examples:", len(train_tokenized))
print("Total multilingual validation examples:", len(validation_tokenized))

print("Training examples:", len(train_tokenized))
print("Validation examples:", len(validation_tokenized))


training_args = TrainingArguments(


    output_dir="./medical_mt5_finetunedMED_EXP",

    learning_rate=1e-4,          # Lower learning rate

    weight_decay=0.01,

    optim="adamw_torch",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=2,

     max_grad_norm=1.0,

    fp16=False,                  # Disable mixed precision

    num_train_epochs=3,

    eval_strategy="epoch",

    save_strategy="steps",
    save_steps=250,

    logging_steps=100
)



data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id= -100,   ###
)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_tokenized,

    eval_dataset=validation_tokenized,


    data_collator=data_collator
)

Total multilingual training examples: 1212
Total multilingual validation examples: 296
Training examples: 1212
Validation examples: 296


In [ ]:
# Start fine tuning process


print("Starting multilingual fine-tuning...")

print(next(model.parameters()).device)

trainer.train()

SAVE_PATH = "/content/drive/MyDrive/MedicalMT5/medical_mt5_finetunedMED_EXP"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Model saved succesfully!")


Starting multilingual fine-tuning...
cuda:0


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,6.118982,2.752522
2,5.594708,2.661146
3,5.373293,2.651039


Model saved succesfully!


In [ ]:
#----------------------------
# Test fine-tuned Medical-mT5
#----------------------------

model.eval()

example = validation_dataset[0]

question = example["full_question"]
reference = example["full_answer"]

prompt = f"""Question:
{question}

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    num_beams=4,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
    early_stopping=True
)

prediction = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("QUESTION:")
print(question)

print("\nMODEL PREDICTION:")
print(prediction)

print("\nREFERENCE ANSWER:")
print(reference)

QUESTION:
A patient diagnosed with bronchial asthma who is being treated with low-dose inhaled steroids (200 micrograms of budesonide every 12 hours) and rescue medication (salbutamol on demand) comes for periodic clinical monitoring. The patient reports that in the last 3 months he has had no exacerbations and that he uses salbutamol once a day. He only has dyspnea when climbing stairs and some nights he sleeps irregularly due to coughing. Which of the following therapeutic options should be considered as the treatment of choice?

MODEL PREDICTION:
When the patient is treated with inhaled steroids and rescue medication (salbutamol once a day) it is advisable to avoid the use of salbutamol.

REFERENCE ANSWER:
Asthma treatment algorithm. We rule out continuing with the same treatment since the patient is symptomatic.


In [ ]:
#----------------------------
# Test 5 validation questions
#----------------------------

model.eval()

for i in range(5):

    example = validation_dataset[i]

    question = example["full_question"]
    reference = example["full_answer"]

    prompt = f"""Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=4,
        repetition_penalty=1.2
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print(f"\n{'='*60}")
    print(f"QUESTION {i+1}")
    print(f"{'='*60}")

    print("\nQUESTION:")
    print(question)

    print("\nPREDICTION:")
    print(prediction)

    print("\nREFERENCE:")
    print(reference)


QUESTION 1

QUESTION:
A patient diagnosed with bronchial asthma who is being treated with low-dose inhaled steroids (200 micrograms of budesonide every 12 hours) and rescue medication (salbutamol on demand) comes for periodic clinical monitoring. The patient reports that in the last 3 months he has had no exacerbations and that he uses salbutamol once a day. He only has dyspnea when climbing stairs and some nights he sleeps irregularly due to coughing. Which of the following therapeutic options should be considered as the treatment of choice?

PREDICTION:
Because of the lack of clinical improvement, it is probably because of the lack of clinical improvement, it is probably because of the lack of clinical improvement and the absence of clinical improvement, it is probably because of the lack of clinical improvement and the absence of clinical improvement, it is probably because of the lack of clinical improvement and the absence of clinical improvement. It is probably because of the la

In [ ]:
#----------------------------
# Verify training labels
#----------------------------

sample = train_tokenized[0]

print("MODEL INPUT:")
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=True
    )
)

print("\nEXPECTED TARGET:")
print(
    tokenizer.decode(
        [
            token for token in sample["labels"]
            if token != -100
        ],
        skip_special_tokens=True
    )
)

MODEL INPUT:
Question: In a 3-month-old infant all the clinical manifestations outlined below would make it necessary to rule out cystic fibrosis, EXCEPT: Answer:

EXPECTED TARGET:
Dehydration may be due to diarrhea of other etiologies. I have doubts about answer 3 since it does not specify which secretions are involved. We have to understand that the growth of Pseudomona aeruginosa is in the culture of bronchial secretions. Because clearly the indication to perform a sweat chlorine (to rule out FQP) are the HIPOnatremic dehydrations and not the hypernatremic ones as the answer says.


In [ ]:
#----------------------------
# Verify corrected labels
#----------------------------

sample = train_tokenized[0]

print("MODEL INPUT:")
print(tokenizer.decode(
    sample["input_ids"],
    skip_special_tokens=True
))

print("\nEXPECTED TARGET:")
print(tokenizer.decode(
    [x for x in sample["labels"] if x != -100],
    skip_special_tokens=True
))

MODEL INPUT:
Question: In a 3-month-old infant all the clinical manifestations outlined below would make it necessary to rule out cystic fibrosis, EXCEPT: Answer:

EXPECTED TARGET:
Dehydration may be due to diarrhea of other etiologies. I have doubts about answer 3 since it does not specify which secretions are involved. We have to understand that the growth of Pseudomona aeruginosa is in the culture of bronchial secretions. Because clearly the indication to perform a sweat chlorine (to rule out FQP) are the HIPOnatremic dehydrations and not the hypernatremic ones as the answer says.


In [ ]:
#----------------------------
# Test on a training example
#----------------------------

import torch

model.eval()

sample = train_tokenized[0]

inputs = {
    "input_ids": torch.tensor(
        [sample["input_ids"]],
        device=model.device
    ),
    "attention_mask": torch.tensor(
        [sample["attention_mask"]],
        device=model.device
    )
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        num_beams=4
    )

prediction = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

reference = tokenizer.decode(
    sample["labels"],
    skip_special_tokens=True
)

print("MODEL PREDICTION:")
print(prediction)

print("\nEXPECTED ANSWER:")
print(reference)

MODEL PREDICTION:
<extra_id_0> In a 3-year-old infant with cystic fibrosis, the cystic fibrosis syndrome is the most common clinical manifestation of cystic fibrosis. In a 3-year-old infant with cystic fibrosis, the cystic fibrosis syndrome is the most common clinical manifestation of cystic fibrosis. In a 3-year-old infant with cystic fibrosis, the cystic fibrosis syndrome is the most common clinical manifestation of cystic fibrosis. In a 3-year-old infant with cystic fibrosis, the cystic fibrosis syndrome is

EXPECTED ANSWER:
Dehydration may be due to diarrhea of other etiologies. I have doubts about answer 3 since it does not specify which secretions are involved. We have to understand that the growth of Pseudomona aeruginosa is in the culture of bronchial secretions. Because clearly the indication to perform a sweat chlorine (to rule out FQP) are the HIPOnatremic dehydrations and not the hypernatremic ones as the answer says.


In [ ]:
import transformers
import peft
import torch

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("PyTorch:", torch.__version__)

Transformers: 4.51.3
PEFT: 0.15.2
PyTorch: 2.11.0+cu130


In [ ]:
%pip install -q --force-reinstall --no-deps "transformers==4.51.3" "peft==0.15.2"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

SAVE_PATH = "/content/drive/MyDrive/MedicalMT5/medical_mt5_finetunedMED_EXP"
model_name = "HiTZ/Medical-mT5-large"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model = PeftModel.from_pretrained(
    base_model,
    SAVE_PATH
)

model = model.to(device)
model.eval()

print("Fine-tuned Medical-mT5 loaded successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'kasa_config', 'lora_ga_config', 'monteclora_config', 'peft_version', 'qalora_group_size', 'target_parameters', 'use_bdlora', 'use_qalora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library a

Fine-tuned Medical-mT5 loaded successfully!


In [ ]:
# ============================================================
# Load fine-tuned Medical-mT5 model
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

SAVE_PATH = "/content/drive/MyDrive/MedicalMT5/medical_mt5_finetunedMED_EXP"

model_name = "HiTZ/Medical-mT5-large"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# Load original tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

# Load saved LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    SAVE_PATH
)

# Move to GPU
model = model.to(device)

# Evaluation mode
model.eval()

print("Fine-tuned Medical-mT5 loaded successfully!")

Using device: cuda
Fine-tuned Medical-mT5 loaded successfully!


In [ ]:
print([name for name in globals() if "validation" in name])

[]


In [ ]:
# ============================================================
# Reload English validation dataset
# ============================================================

from datasets import load_dataset

# Load original English dataset
ds = load_dataset("HiTZ/MedExpQA", "en")

dataset = ds["train"]

# Reproduce original shuffle
dataset = dataset.shuffle(seed=42)

# Original 70% training split
first_split = dataset.train_test_split(
    train_size=0.70,
    seed=42
)

train_dataset = first_split["train"]
remaining_dataset = first_split["test"]

# Original validation/test split
remaining_split = remaining_dataset.train_test_split(
    train_size=17/30,
    seed=42
)

validation_dataset = remaining_split["train"]
test_dataset = remaining_split["test"]

print("Training:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

print("\nFirst validation question:")
print(validation_dataset[0]["full_question"])

Training: 303
Validation: 74
Test: 57

First validation question:
A patient diagnosed with bronchial asthma who is being treated with low-dose inhaled steroids (200 micrograms of budesonide every 12 hours) and rescue medication (salbutamol on demand) comes for periodic clinical monitoring. The patient reports that in the last 3 months he has had no exacerbations and that he uses salbutamol once a day. He only has dyspnea when climbing stairs and some nights he sleeps irregularly due to coughing. Which of the following therapeutic options should be considered as the treatment of choice?


In [ ]:
from datasets import load_dataset

ds = load_dataset("HiTZ/MedExpQA", "en")

print(ds)

DatasetDict({
    train: Dataset({
        features: ['correct_option', 'explanations', 'full_answer', 'full_answer_no_ref', 'full_question', 'id', 'lang', 'options', 'question_id_specific', 'type', 'year', 'rag'],
        num_rows: 434
    })
    validation: Dataset({
        features: ['correct_option', 'explanations', 'full_answer', 'full_answer_no_ref', 'full_question', 'id', 'lang', 'options', 'question_id_specific', 'type', 'year', 'rag'],
        num_rows: 63
    })
    test: Dataset({
        features: ['correct_option', 'explanations', 'full_answer', 'full_answer_no_ref', 'full_question', 'id', 'lang', 'options', 'question_id_specific', 'type', 'year', 'rag'],
        num_rows: 125
    })
})


In [ ]:
# ============================================================
# Generate and save Medical-mT5 evaluation results
# ============================================================

import json
import os
import torch
from tqdm.auto import tqdm

model.eval()

results = []

# Evaluate English validation dataset
for i, example in enumerate(tqdm(validation_dataset)):

    question = example["full_question"]
    reference = example["full_answer"]

    prompt = f"""Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    results.append({
        "question_id": example.get("id", i),
        "language": "en",
        "question": question,
        "prediction": prediction,
        "reference_answer": reference
    })

# ============================================================
# Save results to Google Drive
# ============================================================

SAVE_DIR = "/content/drive/MyDrive/MedicalMT5/results"

os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_FILE = os.path.join(
    SAVE_DIR,
    "medical_mt5_medexpqa_english_results.json"
)

with open(SAVE_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("Results saved successfully!")
print("File:", SAVE_FILE)
print("Total questions evaluated:", len(results))

  0%|          | 0/74 [00:00<?, ?it/s]

Results saved successfully!
File: /content/drive/MyDrive/MedicalMT5/results/medical_mt5_medexpqa_english_results.json
Total questions evaluated: 74


In [ ]:
# ============================================================
# Generate and save Medical-mT5 evaluation results
# ============================================================

import json
import os
import torch
from tqdm.auto import tqdm

model.eval()

results = []

# Evaluate English validation dataset
for i, example in enumerate(tqdm(validation_dataset)):

    question = example["full_question"]
    reference = example["full_answer"]

    prompt = f"""Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    results.append({
        "question_id": example.get("id", i),
        "language": "en",
        "question": question,
        "prediction": prediction,
        "reference_answer": reference
    })

# Save to Google Drive
SAVE_DIR = "/content/drive/MyDrive/MedicalMT5/results"
os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_FILE = os.path.join(
    SAVE_DIR,
    "medical_mt5_medexpqa_english_results.json"
)

with open(SAVE_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("Results saved successfully!")
print("File:", SAVE_FILE)
print("Total questions evaluated:", len(results))

NameError: name 'validation_dataset' is not defined